# Portfolio - Part 2
## All to be checked

### Ex. 1 - WIP

In [ ]:
import pandas as pd

data = {
    "1st It": [2,1,1,1,1],
    "2nd It": [4,3,4,3,2],
    "3rd It": [6,5,9,5,6],
    "4th It": [8,7,16,7,24],
    "5th It": [10,9,25,11,120],
    "6th It": [12,11,36,13,720],
    "7th It": [14,13,49,17,5040],
    "8th It": [16,15,64,19,40320],
    "9th It": [18,17,81,23,362880],
    "10th It": [20,19,100,29,3628800]
}

df = pd.DataFrame(data, index = ["Even #s", "Odd #s", "Square #s", "Prime #s", "Faculty #s"])

#print(df)
print('\n')
print(df.head())
print('\nShape: ')
print(df.shape)
print('\n')
print(df.info())

### Ex. 2

In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

data = sns.load_dataset("iris")

numeric_data = data.select_dtypes(include='number')

plt.figure(figsize=(10, 6))
plt.boxplot(numeric_data.values, tick_labels=numeric_data.columns)
plt.title('Boxplot of each Iris Dataset Feature')
plt.ylabel('cm')
plt.grid(True)
plt.tight_layout()
plt.show()



### Ex. 3

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt
import plotly.express as px

data = sns.load_dataset("tips")

df = pd.DataFrame(data)

fig = px.sunburst(df, path=['sex', 'day', 'time'], values='total_bill')
fig.show()

### Ex. 4 - WIP

In [ ]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib
import matplotlib.pyplot as plt
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split

def rmse(targets, predictions):
    return np.sqrt(np.mean(np.square(predictions - targets)))

data = sns.load_dataset("tips")
df = pd.DataFrame(data)

inputs, targets = df[["total_bill"]], df['tip']

model = LinearRegression().fit(inputs, targets)
guesses = model.predict(inputs)
loss = rmse(targets, guesses)
print('Loss:', loss)

inputs_train, inputs_test, targets_train, targets_test = train_test_split(inputs, targets, test_size=0.1)

model = LinearRegression().fit(inputs_train, targets_train)
predictions_train = model.predict(inputs_train)

model = LinearRegression().fit(inputs_test, targets_test)
predictions_test = model.predict(inputs_test)

tb = 50
quip_tip = {"total_bill": [tb]}

tp = pd.DataFrame(quip_tip)
tip_prediction = model.predict(tp)
print("The predicted tip for a bill of ",tb, "is", round(tip_prediction[0],2))




### Ex. 5 - WIP {Tut8.1}

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report
from sklearn.tree import plot_tree, export_text
import pickle
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import itertools

dataset_path = 'https://raw.githubusercontent.com/Koldim2001/test_api/refs/heads/main/titanic.csv' 
df = pd.read_csv(dataset_path)

df = df[['Survived','Pclass', 'Sex', 'Age', 'SibSp', 'Parch', 'Fare', 'Embarked']]  # The subset (columns) we selected for this project
df = df.dropna(subset=['Age'])
df.drop(columns='Survived')
#df.info()
#print(df.shape)

sex_codes = {'female': 0, 'male': 1}
df['Sex'] = df.Sex.map(sex_codes)

embark_codes = {'S': 0, 'Q': 1, 'C':2}
df['Embarked'] = df.Embarked.map(embark_codes)

train, test = train_test_split(df, test_size=0.2)

def plot_confusion_matrix(cm, classes, normalize=False, title='Confusion matrix', cmap=plt.cm.Blues):
    """
    Plots confusion matrix
    cm - confusion matrix
    classes - class list
    normalize - normalize to 1 if True
    title - plot title
    cmap - color map
    """

    if normalize:
        cm = cm.astype('float') / cm.sum(axis=1)[:, np.newaxis]
        print("Normalized confusion matrix")
    else:
        print('Confusion matrix, without normalization')

    plt.imshow(cm, interpolation='nearest', cmap=cmap)
    plt.title(title)
    plt.colorbar()
    tick_marks = np.arange(len(classes))
    plt.xticks(tick_marks, classes, rotation=45)
    plt.yticks(tick_marks, classes)

    fmt = '.2f' if normalize else 'd'
    thresh = cm.max() / 2.
    for i, j in itertools.product(range(cm.shape[0]), range(cm.shape[1])):
        plt.text(j, i, format(cm[i, j], fmt),
                 horizontalalignment="center",
                 color="white" if cm[i, j] > thresh else "black")

    plt.tight_layout()
    plt.ylabel('True label')
    plt.xlabel('Predicted label')


def experiment(max_depth, min_samples_split):
    """
    Builds and trains Decision Tree model
    """
    # Build and train Decision Tree model
    model = DecisionTreeClassifier(max_depth=max_depth, min_samples_split=min_samples_split, random_state=42)
    model.fit(train.drop('Survived', axis=1), train['Survived'])

    # Calculate accuracy metrics
    preds = model.predict(test.drop('Survived', axis=1))
    acc = accuracy_score(test['Survived'], preds)
    cm = confusion_matrix(test['Survived'], preds)

    print("Accuracy: ", round(acc*100,2), "%")

    # Plot confusion matrix
    plot_confusion_matrix(cm, classes=['Not Survived', 'Survived'])

    # Classification report
    report = classification_report(test['Survived'], preds, target_names=['Not Survived', 'Survived'])
    print(report)

    # Save model in pickle format
    with open('../outputs/models/p2model_dt.pkl', 'wb') as f:
        pickle.dump(model, f)
        
max_depth = 5
min_samples_split = 150

experiment(max_depth, min_samples_split)



In [ ]:
with open('../outputs/models/p2model_dt.pkl', 'rb') as f:
    model = pickle.load(f)


# Predict outcome of Titanic trip for a person
person = pd.DataFrame({
	'Pclass':[3],
    'Sex':[1],
	'Age':[55],
    'SibSp':[0],
    'Parch':[2],
	'Fare':[7.2500],   
    'Embarked':[1]
})

prediction = model.predict(person)
print(f"The model predicts {prediction}")

if prediction == [1]:
    print ("This person is, the most likely, is a survivor.")
else:
    print("This person, the most likely, perished.")


importance_df = pd.DataFrame({
    'Feature': df.drop(columns='Survived').columns,
    'Importance': model.feature_importances_
}).sort_values('Importance', ascending=False)
plt.title('Feature Importance')
sns.barplot(data=importance_df.head(10), x='Importance', y='Feature', hue='Importance')